# Démo 6 : convolution, pooling et feature maps

Objectif : voir ce que fait une convolution sur une image. On fixe d'abord les poids du noyau à la main (flou, bord vertical), puis on vérifie les tailles de sortie selon le padding, le stride et la dilation, on compare max et average pooling, et on affiche des feature maps.

**Données** : Fashion-MNIST (Zalando Research, licence MIT), fourni par torchvision : 70 000 images de vêtements en niveaux de gris, 28 x 28 pixels, 10 classes.

**Ce que vous allez voir**

- ce que fait une convolution, avec des noyaux fixés à la main : flou et détection de bords ;
- comment le padding, le stride et la dilation changent la taille de sortie ;
- le pooling, puis les cartes (feature maps) produites par une vraie couche de convolution.

**Durée prévue** : 10 min.

*Les sorties enregistrées dans ce notebook viennent d'une exécution dans Google Colab sur GPU A100, avec la graine fixée. Sur un autre GPU ou sur CPU, certaines valeurs peuvent légèrement différer : PyTorch ne garantit pas des résultats identiques d'une plateforme à l'autre.*

## 0. Installation

Cette cellule écrit `requirements-colab.txt` (les versions de la formation, alignées sur le runtime Colab), puis l'installe avec uv. Les paquets déjà présents à la bonne version ne sont pas réinstallés. Hors de Colab, elle n'installe rien : l'environnement uv du projet contient déjà ces versions.

In [1]:
import importlib.util
from pathlib import Path

Path("requirements-colab.txt").write_text("""torch==2.11.0
torchvision==0.26.0
numpy==2.1.3
matplotlib==3.10.0
pillow==11.3.0
scikit-learn==1.6.1
pandas==2.2.2; python_version < "3.13"
pandas==2.2.3; python_version >= "3.13"
pyarrow>=14
tensorboard==2.21.0
onnx==1.23.0
onnxruntime==1.30.0
onnxscript==0.7.2
torchserve==0.12.0
torch-model-archiver==0.12.0
""")

if importlib.util.find_spec("google.colab") is not None:
    %pip install -q --progress-bar off uv
    !uv pip install --system -q -r requirements-colab.txt
else:
    print("Hors Colab : environnement uv du projet, rien à installer.")

Dans Colab, seuls les paquets absents du runtime ou à une autre version sont téléchargés. Si torch ou torchvision venaient à être réinstallés, redémarrez la session (**Exécution > Redémarrer la session**) puis relancez le notebook.

## 1. Télécharger Fashion-MNIST

**On télécharge Fashion-MNIST une fois pour toutes dans le dossier `data`.**

Le code :
- `datasets.FashionMNIST("data", train=..., download=True)` : télécharge le jeu d'entraînement, puis celui de
  test, s'ils ne sont pas déjà là ;
- `%%capture` : masque la barre de progression du téléchargement.

In [1]:
%%capture
from torchvision import datasets

datasets.FashionMNIST("data", train=True, download=True)
datasets.FashionMNIST("data", train=False, download=True)

Les fichiers sont dans le dossier `data` : les cellules suivantes les retrouvent sans rien retélécharger.

## 2. Une image de Fashion-MNIST

**On prend une image du jeu de test, pour y appliquer des convolutions.**

On commence par une seule image, pour voir à l'oeil ce que fait chaque opération.

Le code :
- `datasets.FashionMNIST("data", train=False, transform=transforms.ToTensor())` : le jeu de test, chaque image
  convertie en tenseur ;
- `img, label = test_ds[0]` : la première image et son numéro de classe ;
- `img.shape` : (1, 28, 28), c'est-à-dire un **canal** de gris, 28 x 28 pixels. Une image en couleur aurait 3
  canaux (rouge, vert, bleu).

In [2]:
import torch
import torch.nn as nn
from torchvision import datasets, transforms

test_ds = datasets.FashionMNIST("data", train=False, transform=transforms.ToTensor())
img, label = test_ds[0]
print(img.shape, test_ds.classes[label])

L'image est un tenseur `torch.Size([1, 28, 28])` : un canal, 28 x 28 pixels. C'est une bottine, `Ankle boot`.

**À retenir** : la forme d'une image est (canaux, hauteur, largeur).

## 3. Une fonction d'affichage

**Une petite fonction pour afficher plusieurs images côte à côte.**

On va comparer souvent l'image d'origine et le résultat d'une opération.

Le code (à survoler) :
- `afficher(*images)` : accepte un nombre quelconque d'images ;
- `plt.subplots(1, len(images))` : une case par image ; `imshow(..., cmap="gray")` affiche en niveaux de gris.

In [3]:
import matplotlib.pyplot as plt


def afficher(*images):
    fig, axes = plt.subplots(1, len(images), figsize=(2.5 * len(images), 2.5), squeeze=False)
    for ax, image in zip(axes[0], images):
        ax.imshow(image, cmap="gray")
        ax.axis("off")
    plt.show()

Rien à afficher : la fonction est définie. Elle servira à chaque étape.

## 4. Afficher l'image

**L'image d'origine.**

Le code :
- `img[0]` : le canal unique de l'image, un tableau 28 x 28.

In [4]:
afficher(img[0])

La bottine, en blanc sur fond noir, avec sa semelle plus claire en bas.

## 5. Un noyau de flou

**Un noyau de convolution est une petite grille de poids qu'on fait glisser sur l'image.**

En chaque position, la convolution multiplie les pixels couverts par les poids du noyau et fait la
somme. Avec neuf poids égaux à 1/9, cette somme est la moyenne du voisinage : l'image devient floue. Dans un
CNN, ces poids sont appris ; ici, on les fixe à la main pour voir leur effet.

Le code :
- `nn.Conv2d(1, 1, kernel_size=3, padding=1)` : 1 canal en entrée, 1 en sortie, un noyau 3 x 3 ; `padding=1`
  ajoute une bordure de zéros pour garder la taille ;
- `conv_flou.weight.fill_(1.0 / 9.0)` et `bias.zero_()` : on fixe les poids, sous `torch.no_grad()` ;
- `conv_flou(img)` : on applique la convolution.

Analogie : une petite loupe carrée qu'on promène sur toute l'image, et qui fait la moyenne de ce qu'elle voit.

In [5]:
conv_flou = nn.Conv2d(1, 1, kernel_size=3, padding=1)
with torch.no_grad():
    conv_flou.weight.fill_(1.0 / 9.0)
    conv_flou.bias.zero_()
    flou = conv_flou(img)
afficher(img[0], flou[0])

À droite, l'image garde sa taille, grâce à `padding=1`, mais les contours sont adoucis : chaque pixel est devenu la moyenne de ses 9 voisins.

**À retenir** : le noyau décide de l'effet ; ici, fixé à la main, il floute.

## 6. Un noyau qui détecte les bords verticaux

**Un autre noyau : -1 à gauche, 0 au centre, +1 à droite.**

Ce noyau calcule la différence entre la droite et la gauche du pixel. Dans une zone uniforme, la
différence est nulle ; sur un bord vertical, elle est forte. Un noyau est donc un **détecteur de motif**.

Le code :
- `conv_bord.weight[:] = torch.tensor([[-1, 0, 1], ...])` : les neuf poids, ligne par ligne ;
- `.abs()` : on garde la valeur absolue, pour voir aussi les bords qui vont du clair au sombre.

Prédisez : quelles parties de la bottine vont ressortir ?

In [6]:
conv_bord = nn.Conv2d(1, 1, kernel_size=3, padding=1)
with torch.no_grad():
    conv_bord.weight[:] = torch.tensor([[-1.0, 0.0, 1.0], [-1.0, 0.0, 1.0], [-1.0, 0.0, 1.0]])
    conv_bord.bias.zero_()
    bords = conv_bord(img).abs()
afficher(img[0], bords[0])

À droite, seuls les bords ressortent : le contour gauche du talon, le haut de la tige, le bord droit de la bottine. Les zones uniformes, fond noir ou intérieur de la chaussure, donnent une réponse proche de zéro.

**À retenir** : un noyau est un détecteur de motif ; un CNN apprend lui-même les noyaux utiles à sa tâche.

**Essayez** : transposez le noyau (`[[-1, -1, -1], [0, 0, 0], [1, 1, 1]]`) pour détecter les bords horizontaux.

## 7. Padding, stride et dilation : les tailles de sortie

**Trois réglages changent la taille de sortie d'une convolution.**

Pour empiler des couches, il faut savoir quelle taille sort de chacune.
- **padding** : ajoute une bordure de zéros ; avec `padding=1` et un noyau 3 x 3, la taille est conservée ;
- **stride** : le pas du noyau ; avec `stride=2`, une position sur deux, la sortie est deux fois plus petite ;
- **dilation** : écarte les poids du noyau ; il couvre une zone plus large avec autant de poids.

Le code :
- `nn.Conv2d(1, 8, kernel_size=3, **reglage)` : 8 noyaux, donc 8 canaux en sortie ;
- la boucle essaie quatre réglages et affiche la forme obtenue.

Prédisez chaque taille avant d'exécuter.

In [7]:
for reglage in [dict(padding=0), dict(padding=1), dict(padding=1, stride=2), dict(padding=0, dilation=2)]:
    conv = nn.Conv2d(1, 8, kernel_size=3, **reglage)
    print(reglage, "->", tuple(conv(img).shape))

Sans padding, le noyau 3 x 3 perd un pixel de chaque côté : 26 x 26. Avec `padding=1`, la taille est conservée : 28 x 28. Avec `stride=2`, une position sur deux : 14 x 14. Avec `dilation=2`, le noyau couvre une zone 5 x 5 : 24 x 24. Dans tous les cas, 8 canaux en sortie, un par noyau.

**À retenir** : avec un noyau 3 x 3, `padding=1` garde la taille ; c'est le réglage le plus courant.

## 8. Max pooling et average pooling

**Le pooling résume chaque bloc de 2 x 2 pixels par une seule valeur.**

Il réduit la taille des cartes, donc le calcul des couches suivantes, et rend le réseau moins
sensible à un petit décalage de l'image. Il n'a aucun paramètre à apprendre.

Le code :
- `nn.MaxPool2d(2)` : garde le maximum de chaque bloc, la réponse la plus forte ;
- `nn.AvgPool2d(2)` : garde la moyenne, ce qui lisse ;
- on affiche les bords, puis les deux résultats.

In [8]:
maxi = nn.MaxPool2d(2)(bords)
moyenne = nn.AvgPool2d(2)(bords)
print(tuple(bords.shape), "->", tuple(maxi.shape))
afficher(bords[0], maxi[0], moyenne[0])

Le pooling divise la taille par deux, de 28 x 28 à 14 x 14, sans aucun paramètre. Le max pooling (au milieu) garde les bords les plus marqués ; l'average pooling (à droite) les atténue un peu.

**À retenir** : le pooling réduit la taille, donc le calcul, sans rien apprendre.

## 9. Les feature maps d'une couche de convolution

**Une vraie couche de convolution, avec 8 noyaux, produit 8 cartes.**

Chaque carte (feature map) montre où l'image contient le motif que cherche un noyau. Les poids sont
encore aléatoires ; l'entraînement les transformera en détecteurs utiles.

Le code :
- `nn.Conv2d(1, 8, kernel_size=3, padding=1)` : 8 noyaux 3 x 3 aux poids tirés au hasard ;
- `torch.relu(...)` : l'activation, qui met à zéro les réponses négatives ;
- `afficher(*cartes)` : les 8 cartes côte à côte.

In [ ]:
torch.manual_seed(0)
with torch.no_grad():
    cartes = torch.relu(nn.Conv2d(1, 8, kernel_size=3, padding=1)(img))
afficher(*cartes)

Huit cartes, une par noyau aléatoire : certaines font ressortir les contours, d'autres la surface de la bottine, une autre presque rien, car la ReLU met à zéro les réponses négatives. L'entraînement spécialisera ces noyaux.

**À retenir** : une couche de convolution produit autant de cartes que de noyaux.

## À retenir de cette démo

- Une convolution fait glisser un petit noyau sur l'image ; ses poids décident du motif détecté (flou, bord...). Dans un CNN, ces poids sont appris.
- Padding, stride et dilation règlent la taille de sortie ; le pooling divise la taille par deux, sans paramètre.
- Une couche `Conv2d(1, 8)` produit 8 cartes, une par noyau : autant de détecteurs de motifs.